# Disaster Impact Pipeline: Python to MySQL

This notebook fetches data from three sources, cleans it, and loads it into a MySQL database:
- **FEMA** Disaster Declarations (REST API)
- **Census ACS 2022** Demographics (REST API)
- **HUD FY2025** Fair Market Rents (local CSV)

**Prerequisites**: MySQL server running locally, `cpsc5071` conda kernel selected.

The key assumption is that county-level demographic and housing cost characteristics remain relatively stable year-to-year. This allows us to combine point-in-time snapshots (Census 2022, HUD FY2025) with multi-year disaster history (FEMA 2020-2024) in a cross-sectional framework.

Our goal is to identify spatial patterns—which types of counties (characterized by their demographics and housing costs) tend to receive more federal disaster assistance—rather than to forecast temporal patterns (predicting disasters in specific future years).

In other words, we're answering: "What county characteristics predict higher disaster aid allocation?" not "When will the next disaster occur?" This approach assumes that the relationship between county characteristics and disaster outcomes remains consistent across the observation period, which is standard practice in cross-sectional disaster impact research.

# ***Take a step back before taking two steps forward:***
Before starting this notebook, please review our preprocessing notebook (feel free to take a look [here](https://github.com/dcnguyen060899/su_cpsc5071_group_project_sad_sravya_anushka_duy/blob/main/code/initlal_data_exploratory.ipynb)). After examining a sample of the dataset to understand the data, we analyzed the raw data sources. Using these preprocessing insights, we documented our initial plan for the database design (take a look [here](https://drive.google.com/file/d/1cJLC4h_BQLGuLkaGgxAhy0MhS38nj7_v/view?usp=sharing)). You can read through that document to get a sense of how we leveraged our initial understanding of the raw data;  by understanding what problems it faces, and how we transformed that understanding to help us obey database management constraints and overall rules we learned in class for a clean, efficient design that still preserves the original information. Finally, we documented our official design prior to writing this script (feel free to take a look [here](https://drive.google.com/file/d/1U61kwf9-XdRTGRhHI67mdP4fKEjfKaUb/view?usp=sharing)). Reading that document would help you understand our pedagogical reasoning and approach to designing and implementing this database. Note that these are extra readings not included in the repository. As we wrote them, we help the readers and ourself to solidify our foundational understanding so we can mitigate errors in our database that could potentially affect answering our research question.

# Let's begin:

## Imports & MySQL Connection

In [ ]:
import mysql.connector
from mysql.connector import Error
import requests
import pandas as pd
import numpy as np

# Creating a connection helper (course pattern that we learned from W6) ──
def create_connection(host_name, user_name, user_password, db_name=None):
    connection = None
    try:
        connection = mysql.connector.connect(
            host=host_name,
            user=user_name,
            passwd=user_password,
            database=db_name
        )
        print("Connection to MySQL DB successful")
    except Error as e:
        print(f"The error '{e}' occurred")
    return connection

def execute_query(connection, query):
    cursor = connection.cursor()
    try:
        cursor.execute(query)
        connection.commit()
        print("Query executed successfully")
    except Error as e:
        print(f"The error '{e}' occurred")

def read_query(connection, query):
    cursor = connection.cursor()
    try:
        cursor.execute(query)
        return cursor.fetchall()
    except Error as e:
        print(f"The error '{e}' occurred")
        return None

# Connection config
# It is important to replace password with your own MySQL password, please do not use my password, that's for my computer, not your
config = {
    'host': 'localhost',
    'user': '', # input your own local username for your database
    'password': '', # input your own local database password
    'port': 3306
}

# Create database
connection = create_connection(config['host'], config['user'], config['password'])
execute_query(connection, "CREATE DATABASE IF NOT EXISTS disaster_impact_db")
connection.close()

# Reconnect with database selected
connection = create_connection(config['host'], config['user'], config['password'], 'disaster_impact_db')
print("Database disaster_impact_db ready.")

## Create Tables (DDL)

In [ ]:
# please drop existing tables in reverse FK order for clean reruns
for tbl in ['housing_costs', 'demographics', 'disasters', 'county']:
    execute_query(connection, f"DROP TABLE IF EXISTS {tbl}")

# ── Dimension: county lookup ──
execute_query(connection, """
CREATE TABLE IF NOT EXISTS county (
    fips CHAR(5) PRIMARY KEY,
    county_name VARCHAR(100),
    state_abbr CHAR(2)
)
""")

# ── Fact: disaster events ──
execute_query(connection, """
CREATE TABLE IF NOT EXISTS disasters (
    id VARCHAR(40) PRIMARY KEY,
    fips CHAR(5),
    disaster_number INT,
    declaration_type CHAR(2),
    declaration_date DATE,
    fy_declared INT,
    incident_type VARCHAR(50),
    declaration_title VARCHAR(200),
    incident_begin_date DATE,
    incident_end_date DATE,
    ih_program BOOLEAN,
    ia_program BOOLEAN,
    pa_program BOOLEAN,
    hm_program BOOLEAN,
    FOREIGN KEY (fips) REFERENCES county(fips)
)
""")

# ── Dimension: demographics (2022 ACS) ──
execute_query(connection, """
CREATE TABLE IF NOT EXISTS demographics (
    fips CHAR(5) PRIMARY KEY,
    county_name VARCHAR(100),
    total_population INT,
    median_household_income INT,
    poverty_count INT,
    unemployment_count INT,
    labor_force_count INT,
    FOREIGN KEY (fips) REFERENCES county(fips)
)
""")

# ── Dimension: housing costs (FY2025 FMR) ──
execute_query(connection, """
CREATE TABLE IF NOT EXISTS housing_costs (
    fips CHAR(5) PRIMARY KEY,
    county_name VARCHAR(100),
    metro BOOLEAN,
    pop2022 INT,
    fmr_0 INT,
    fmr_1 INT,
    fmr_2 INT,
    fmr_3 INT,
    fmr_4 INT,
    FOREIGN KEY (fips) REFERENCES county(fips)
)
""")

print("\nAll 4 tables created.")

## Fetch & Clean FEMA Disaster Data (2020-2024)

In [ ]:
fema_url = "https://www.fema.gov/api/open/v2/DisasterDeclarationsSummaries"

all_records = []
skip = 0
page_size = 1000

print("Fetching FEMA disaster data (2020-2024)...")
while True:
    params = {
        '$filter': 'fyDeclared ge 2020',
        '$top': page_size,
        '$skip': skip
    }
    resp = requests.get(fema_url, params=params, timeout=60)
    resp.raise_for_status()
    records = resp.json()['DisasterDeclarationsSummaries']
    if not records:
        break
    all_records.extend(records)
    print(f"  Fetched {len(all_records)} records so far...")
    skip += page_size

df_fema = pd.DataFrame(all_records)
print(f"\nTotal FEMA records fetched: {len(df_fema)}")

# Build 5-digit FIPS
df_fema['fips'] = df_fema['fipsStateCode'].astype(str).str.zfill(2) + \
                   df_fema['fipsCountyCode'].astype(str).str.zfill(3)

# Parse dates: strip ISO suffix
date_cols = ['declarationDate', 'incidentBeginDate', 'incidentEndDate']
for col in date_cols:
    df_fema[col] = df_fema[col].apply(
        lambda x: x[:10] if isinstance(x, str) and len(x) >= 10 else None
    )

# Keep only columns we need
df_fema_clean = df_fema[[
    'id', 'fips', 'disasterNumber', 'declarationType',
    'declarationDate', 'fyDeclared', 'incidentType', 'declarationTitle',
    'incidentBeginDate', 'incidentEndDate',
    'ihProgramDeclared', 'iaProgramDeclared',
    'paProgramDeclared', 'hmProgramDeclared'
]].copy()

df_fema_clean.columns = [
    'id', 'fips', 'disaster_number', 'declaration_type',
    'declaration_date', 'fy_declared', 'incident_type', 'declaration_title',
    'incident_begin_date', 'incident_end_date',
    'ih_program', 'ia_program', 'pa_program', 'hm_program'
]

print(f"Cleaned FEMA shape: {df_fema_clean.shape}")
print(f"Unique FIPS in FEMA: {df_fema_clean['fips'].nunique()}")
df_fema_clean.head()

## Fetch & Clean Census ACS 2022 Data

In [ ]:
census_url = "https://api.census.gov/data/2022/acs/acs5"

variables = {
    'B01001_001E': 'total_population',
    'B19013_001E': 'median_household_income',
    'B17001_002E': 'poverty_count',
    'B23025_005E': 'unemployment_count',
    'B23025_003E': 'labor_force_count'
}

params = {
    'get': 'NAME,' + ','.join(variables.keys()),
    'for': 'county:*',
    'in': 'state:*'
}

print("Fetching Census ACS 2022 data...")
resp = requests.get(census_url, params=params, timeout=60)
resp.raise_for_status()
data = resp.json()

headers = data[0]
df_census = pd.DataFrame(data[1:], columns=headers)
print(f"Census records fetched: {len(df_census)}")

# Build 5-digit FIPS
df_census['fips'] = df_census['state'].str.zfill(2) + df_census['county'].str.zfill(3)

# Convert numeric columns
for var_code in variables.keys():
    df_census[var_code] = pd.to_numeric(df_census[var_code], errors='coerce')

# Replace Census missing sentinel (-666666666) with NaN
for var_code in variables.keys():
    df_census.loc[df_census[var_code] == -666666666, var_code] = np.nan

# Build clean DataFrame
df_census_clean = df_census[['fips', 'NAME'] + list(variables.keys())].copy()
df_census_clean.columns = ['fips', 'county_name'] + list(variables.values())

# Convert to nullable int (keeps NaN while allowing int display)
for col in variables.values():
    df_census_clean[col] = df_census_clean[col].astype('Int64')

print(f"Cleaned Census shape: {df_census_clean.shape}")
print(f"Unique FIPS: {df_census_clean['fips'].nunique()}")
print(f"Missing median_household_income: {df_census_clean['median_household_income'].isna().sum()}")
df_census_clean.head()

## Clean HUD Fair Market Rents Data

Since in the data exploration script, we found out that because of county town where many could belong to the same county, we get duplicate fips. Another problem were that the fair market rent from each town evaluate differently but again, they belong to the same county. Last problem we found were that due to the key constraint, the DBMS expect our primary key fips to be unique, but if we don't address the two problem above and naively extract the 5-digit FIPS, we would try to insert multiple rows with the same FIPS → Primary key violation!

So we are proposing the following:
1. Group all town rows by their 5-digit county FIPS
2. Average the FMR values, weighted by town population
3. Town with 50,000 people has more influence than town with 500 people
4. Set metro = 1 if any town in the county is metro
5. Sum the populations across all towns



In [ ]:
hud_path = '../data/FY25_FMRs-Table 1.csv'
df_hud = pd.read_csv(hud_path)
print(f"Raw HUD shape: {df_hud.shape}")

# Extract 5-digit county FIPS from the 10-digit FIPS code
# HUD fips: e.g. 0100199999 → first 5 digits = 01001
df_hud['county_fips'] = df_hud['fips'].apply(lambda x: str(x).zfill(10)[:5])

print(f"Unique county FIPS before dedup: {df_hud['county_fips'].nunique()}")
print(f"Rows with county_town_name (NE subdivisions): {df_hud['county_town_name'].notna().sum()}")

# Aggregate to county level using population-weighted averages for FMR columns
fmr_cols = ['fmr_0', 'fmr_1', 'fmr_2', 'fmr_3', 'fmr_4']

def weighted_agg(group):
    total_pop = group['pop2022'].sum()
    row = {
        'county_name': group['countyname'].iloc[0],
        'metro': int(group['metro'].max()),  # metro if any sub-area is metro
        'pop2022': total_pop,
    }
    for col in fmr_cols:
        if total_pop > 0:
            row[col] = int(round((group[col] * group['pop2022']).sum() / total_pop))
        else:
            row[col] = int(group[col].iloc[0])
    return pd.Series(row)

df_hud_clean = df_hud.groupby('county_fips').apply(weighted_agg, include_groups=False).reset_index()
df_hud_clean.rename(columns={'county_fips': 'fips'}, inplace=True)

print(f"\nCleaned HUD shape: {df_hud_clean.shape}")
print(f"Unique FIPS: {df_hud_clean['fips'].nunique()}")
df_hud_clean.head()

note that if we don't average the fmr across town, we are violating the key constraint. For example:

If FEMA has 2 disaster rows for county 09001 (Washington County example style).

HUD has 5 housing rows for the same FIPS (five towns).

Then when we querry:
```SELECT *
FROM FEMA_Disaster d
JOIN HousingCosts h
  ON d.fips = h.fips;
```
then 2 disasters × 5 HUD rows = 10 joined rows.

Each original disaster appears 5 times with different FMR values.

If we now count disasters per county, we wrongly think there are 10 disasters, not 2.

Therefore this is a many to many explosion problem, meaning that one county is connecting to many HUD rows and after we join without realized this one to one fips grain mismatched, we now get one HUDS fips links to multiple fips FEMA fips.

Well of course, obviously, we learned in class, we can use a junction table, but however, if we include the town level, it would defeat our county level research question trying to investigate hence population weighted average is the way to go


## Build & Insert County Dimension Table

Note that all these three different definitions of county boundaries create a logical inconsistency. Imagine if we simply stored FEMA data in a disasters table, Census data in a demographics table, and HUD data in a housing costs table without creating a county table, the database would have no authoritative answer to the question of which FIPS codes are valid. The FIPS code 01001 appears in all three sources and clearly represents Autauga County, Alabama. However, some FIPS codes appear in one or two sources but not all three. Without a county table, the database cannot distinguish between a FIPS code that legitimately represents a geographic entity absent from one source versus a FIPS code that represents a data error or invalid code.

So we propose to add a fourth table called county table such that it establish a single authoritative registry of all valid FIPS codes across our integrated dataset. That mean the county table does not correspond to any single raw dataset, it represents a logical concept, the set of all geographic entities that participate in your analysis, rather than a physical data source. The county table is constructed through a union operation that combines the unique FIPS codes from all three sources, ensuring that the dimension encompasses every entity mentioned anywhere in your database.


In [ ]:
# Union all unique FIPS from Census, HUD, and FEMA
all_fips = set(df_census_clean['fips'].unique())
all_fips |= set(df_hud_clean['fips'].unique())
all_fips |= set(df_fema_clean['fips'].unique())
print(f"Total unique FIPS codes across all sources: {len(all_fips)}")

# Build county dimension from Census names (most complete source for names)
# Census NAME format: "Autauga County, Alabama" → extract county_name and state
census_lookup = df_census_clean[['fips', 'county_name']].copy()

# Also get state abbreviations from FEMA data (Census doesn't have abbreviations directly)
fema_state_lookup = df_fema[['fips', 'state']].drop_duplicates('fips').set_index('fips')['state']

# HUD has state abbreviations too
hud_state_lookup = df_hud[['county_fips', 'stusps']].drop_duplicates('county_fips')
hud_state_lookup = hud_state_lookup.set_index('county_fips')['stusps']

# Build county rows
county_rows = []
census_dict = dict(zip(df_census_clean['fips'], df_census_clean['county_name']))
hud_name_dict = dict(zip(df_hud_clean['fips'], df_hud_clean['county_name']))

for fips in sorted(all_fips):
    name = census_dict.get(fips) or hud_name_dict.get(fips) or 'Unknown'
    # Get state abbreviation: try FEMA first, then HUD
    state_abbr = fema_state_lookup.get(fips) or hud_state_lookup.get(fips)
    if state_abbr is None and fips in census_dict:
        # Extract from Census NAME: "County, State" → use state FIPS to abbr mapping
        state_abbr = None  # will be filled below
    county_rows.append((fips, name, state_abbr))

# Insert into county table
cursor = connection.cursor()
insert_county = "INSERT IGNORE INTO county (fips, county_name, state_abbr) VALUES (%s, %s, %s)"
cursor.executemany(insert_county, county_rows)
connection.commit()
print(f"Inserted {cursor.rowcount} rows into county table")
cursor.close()

After running this code, we observed that we extracted 3277 unique county FIPS codes from FEMA data. When we examined the Census data, you found 3,222 unique county FIPS codes. When we examined the HUD data after aggregation, you found 3,228 unique county FIPS codes. Taking the union of these three sets produced 3,292 unique FIPS codes, which became the rows of our county table. The county table is not redundant with any source dataset because it represents the union rather than duplicating any individual source.


## Insert Data into MySQL

In [ ]:
# ── Insert demographics ──
print("Inserting demographics...")
cursor = connection.cursor()
insert_demo = """
INSERT IGNORE INTO demographics
    (fips, county_name, total_population, median_household_income,
     poverty_count, unemployment_count, labor_force_count)
VALUES (%s, %s, %s, %s, %s, %s, %s)
"""
demo_rows = []
for _, row in df_census_clean.iterrows():
    demo_rows.append((
        row['fips'],
        row['county_name'],
        None if pd.isna(row['total_population']) else int(row['total_population']),
        None if pd.isna(row['median_household_income']) else int(row['median_household_income']),
        None if pd.isna(row['poverty_count']) else int(row['poverty_count']),
        None if pd.isna(row['unemployment_count']) else int(row['unemployment_count']),
        None if pd.isna(row['labor_force_count']) else int(row['labor_force_count'])
    ))
cursor.executemany(insert_demo, demo_rows)
connection.commit()
print(f"  Inserted {cursor.rowcount} rows into demographics")
cursor.close()

# ── Insert housing_costs ──
print("Inserting housing_costs...")
cursor = connection.cursor()
insert_hc = """
INSERT IGNORE INTO housing_costs
    (fips, county_name, metro, pop2022, fmr_0, fmr_1, fmr_2, fmr_3, fmr_4)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
"""
hc_rows = []
for _, row in df_hud_clean.iterrows():
    hc_rows.append((
        row['fips'],
        row['county_name'],
        int(row['metro']),
        int(row['pop2022']),
        int(row['fmr_0']), int(row['fmr_1']), int(row['fmr_2']),
        int(row['fmr_3']), int(row['fmr_4'])
    ))
cursor.executemany(insert_hc, hc_rows)
connection.commit()
print(f"  Inserted {cursor.rowcount} rows into housing_costs")
cursor.close()

# ── Insert disasters ──
print("Inserting disasters...")
cursor = connection.cursor()
insert_dis = """
INSERT IGNORE INTO disasters
    (id, fips, disaster_number, declaration_type, declaration_date,
     fy_declared, incident_type, declaration_title,
     incident_begin_date, incident_end_date,
     ih_program, ia_program, pa_program, hm_program)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""
dis_rows = []
for _, row in df_fema_clean.iterrows():
    dis_rows.append((
        row['id'],
        row['fips'],
        int(row['disaster_number']),
        row['declaration_type'],
        row['declaration_date'],
        int(row['fy_declared']),
        row['incident_type'],
        row['declaration_title'][:200] if row['declaration_title'] else None,
        row['incident_begin_date'],
        row['incident_end_date'],
        bool(row['ih_program']),
        bool(row['ia_program']),
        bool(row['pa_program']),
        bool(row['hm_program'])
    ))
cursor.executemany(insert_dis, dis_rows)
connection.commit()
print(f"  Inserted {cursor.rowcount} rows into disasters")
cursor.close()

print("\nAll data inserted.")

## Create Analysis Views

In [ ]:
# Drop views if they exist (for clean reruns)
execute_query(connection, "DROP VIEW IF EXISTS analysis_ready")
execute_query(connection, "DROP VIEW IF EXISTS county_disaster_summary")

# ── View: county-level disaster summary ──
execute_query(connection, """
CREATE VIEW county_disaster_summary AS
SELECT d.fips,
       COUNT(*) AS total_disasters,
       SUM(d.ia_program) AS ia_count,
       SUM(d.pa_program) AS pa_count,
       SUM(d.hm_program) AS hm_count
FROM disasters d
GROUP BY d.fips
""")

# ── View: full analysis-ready dataset (joins all tables) ──
execute_query(connection, """
CREATE VIEW analysis_ready AS
SELECT c.fips, c.county_name, c.state_abbr,
       COALESCE(ds.total_disasters, 0) AS total_disasters,
       COALESCE(ds.ia_count, 0) AS ia_count,
       COALESCE(ds.pa_count, 0) AS pa_count,
       dem.total_population, dem.median_household_income,
       dem.poverty_count, dem.unemployment_count, dem.labor_force_count,
       h.fmr_0, h.fmr_1, h.fmr_2, h.fmr_3, h.fmr_4, h.metro
FROM county c
LEFT JOIN county_disaster_summary ds ON c.fips = ds.fips
LEFT JOIN demographics dem ON c.fips = dem.fips
LEFT JOIN housing_costs h ON c.fips = h.fips
""")

print("\nViews created successfully.")

## Verify & Load into Pandas

In [ ]:
# ── Row counts ──
tables = ['county', 'disasters', 'demographics', 'housing_costs']
print("=" * 40)
print("TABLE ROW COUNTS")
print("=" * 40)
for tbl in tables:
    result = read_query(connection, f"SELECT COUNT(*) FROM {tbl}")
    print(f"  {tbl}: {result[0][0]:,} rows")

# ── Load analysis_ready into pandas via SQLAlchemy ──
from sqlalchemy import create_engine

engine = create_engine(
    f"mysql+mysqlconnector://{config['user']}:{config['password']}@{config['host']}:{config['port']}/disaster_impact_db"
)

df_analysis = pd.read_sql("SELECT * FROM analysis_ready", engine)

print(f"\nanalysis_ready shape: {df_analysis.shape}")
print(f"\nFirst 5 rows:")
print(df_analysis.head())

print(f"\nDescribe:")
print(df_analysis.describe())

# ── Sanity checks ──
print("\n" + "=" * 40)
print("SANITY CHECKS")
print("=" * 40)
print(f"Counties with disasters: {(df_analysis['total_disasters'] > 0).sum()}")
print(f"Counties without disasters: {(df_analysis['total_disasters'] == 0).sum()}")
print(f"\nNULL counts per column:")
print(df_analysis.isnull().sum())

# Cleanup
engine.dispose()
connection.close()
print("\nDone! Connection closed.")

NOTE that we choose LEFT JOIN as you can see, there are some missing data however, preserved our pattern.

Total counties: 3,292

- Counties with disasters: 3,277 (from FEMA)

- Counties with demographics: 3,222 (from Census)

- Counties with housing: 3,228 (from HUD)

Gaps:

- 70 counties in FEMA/HUD but NOT in Census → NULL demographics

- 64 counties in FEMA/Census but NOT in HUD → NULL housing

- 15 counties in Census/HUD but NOT in FEMA → total_disasters = 0

#### Alternatively, if you want to try out:

In principle, we could model the raw HUD structure at the sub‑county level using a separate TownOrMetro entity and a junction table. For example, we might define entities County (5‑digit county FIPS) and TownOrMetro (HUD geographic area), with a CountyTown junction table (fips, hud_area_id) and store HousingCosts at the TownOrMetro grain. That design would support a different research question such as: “Within a county, do towns with higher FMRs experience more severe disaster impacts?”, and we could then create a SQL view that joins FEMA disasters to TownOrMetro and aggregates or allocates county‑level disasters down to towns. However, our current study is explicitly county‑level, with disasters, demographics, and housing all interpreted at the county grain. Allocating county‑level FEMA impacts to towns would require strong assumptions and would deviate from the scope of this project. For this reason, we retain the full HUD sub‑county rows in the raw table for future flexibility but build and use a county‑level HUD view that aggregates town FMRs to a single record per county via population‑weighted averages, ensuring a clean one‑to‑one relationship between County and HousingCosts for our main analysis.
